# Notebook 17: Adapter OOD/OE Recovery

Bu notebook sekiz adapter icin recovery kampanyasini tek akista calistirir.

Akis: repo ve veri erisimi -> OOD/OE exact-overlap preflight -> mevcut readiness -> kosullu Stage B -> kosullu Stage C -> kosullu Stage D -> final 8/8 promotion ozeti -> GitHub push.

Stage gate'leri atlanmaz. Eksik OOD slice, classification fail veya evidence-integrity hatasi varsa ilgili hedef `blocked` olarak raporlanir; final OOD test tuning icin kullanilmaz.

Colab oturumu yarida kesilirse notebook'u yeniden calistirin. Tamamlanan run'lar GitHub'a pushlandigi icin en yeni readiness artefaktlari okunur ve gecilmis gate'ler tekrar calistirilmaz.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token
    os.environ['AADS_GITHUB_RELEASE_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
NOTEBOOK17_SPARSE_PATHS = (
    'README.md', 'PROJECT_STATE.md', 'docs', 'src', 'scripts', 'config', 'colab_notebooks',
    'requirements.txt', 'requirements_colab.txt', 'pyproject.toml',
    'data/prepared_runtime_datasets', 'data/ood_dataset', 'data/oe_dataset',
    'runs/apricot/fruit/apricot_fruit_2026-05-26_09-59-51',
    'runs/apricot/leaf/apricot_leaf_2026-07-13_17-40-52',
    'runs/grape/fruit/grape_fruit_2026-07-13_18-07-48',
    'runs/grape/leaf/grape_leaf_2026-05-26_10-09-10',
    'runs/strawberry/fruit/strawberry_fruit_2026-05-14_11-27-26',
    'runs/strawberry/leaf/strawberry_leaf_2026-07-13_17-49-47',
    'runs/tomato/fruit/tomato_fruit_2026-05-26_10-34-27',
    'runs/tomato/leaf/tomato_leaf_2026-07-13_19-25-54',
)

def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', REPO_URL, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK17_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

ROOT = _ensure_aads_repo_on_path()
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


In [ ]:
NOTEBOOK_NAME = '17_adapter_ood_oe_recovery.ipynb'
NOTEBOOK_FILENAME = '17_adapter_ood_oe_recovery.executed.ipynb'

RECOVERY_CAMPAIGN_PATH = 'docs/architecture/adapter_ood_oe_recovery_campaign.json'
RECOVERY_CONTINUE_ON_ERROR = True
RECOVERY_AUTO_PUSH_TO_GITHUB = True
RECOVERY_AUTO_DISCONNECT_RUNTIME = True
RECOVERY_AUTO_DISCONNECT_GRACE_SECONDS = 30
RECOVERY_LOW_RESOURCE_MODE = True
RECOVERY_TARGETS = []  # e.g. ['strawberry__leaf'] for the required no-push smoke
RECOVERY_RESUME_FROM_LEDGER = True
RECOVERY_USE_GITHUB_DATASET_RELEASE = True  # Phase 9 immutable full-release parity.
RECOVERY_LOCAL_DATASET_ROOT = 'data/prepared_runtime_datasets'
DATASET_RELEASE_REPOSITORY = 'EfeErim/bitirmeprojesi'
DATASET_RELEASE_TAG = 'aads-dataset-v1.0.0'
DATASET_RELEASE_CACHE_ROOT = '.runtime_tmp/dataset_release_cache'
RECOVERY_EVIDENCE_MANIFEST_RELATIVE_PATH = 'adapter_ood_oe_evidence_manifest.csv'
RECOVERY_MAX_COMPLETED_EXPERIMENTS = 0
RECOVERY_BATCH_SIZE = 16
RECOVERY_GRAD_ACCUM_STEPS = 2
RECOVERY_NUM_WORKERS = 2
RECOVERY_PREFETCH = 2
RECOVERY_MIN_FREE_DISK_GIB = 12.0
RECOVERY_RECLAIM_FAILED_RUN_PAYLOADS = True

print('[RECOVERY] campaign=', RECOVERY_CAMPAIGN_PATH)
print('[RECOVERY] continue_on_error=', RECOVERY_CONTINUE_ON_ERROR)
print('[RECOVERY] auto_push=', RECOVERY_AUTO_PUSH_TO_GITHUB)
print('[RECOVERY] low_resource=', RECOVERY_LOW_RESOURCE_MODE, 'max_completed=', RECOVERY_MAX_COMPLETED_EXPERIMENTS)
print('[RECOVERY] targets=', RECOVERY_TARGETS or 'all', 'resume_from_ledger=', RECOVERY_RESUME_FROM_LEDGER)
print('[RECOVERY] dataset_mode=', 'github_release' if RECOVERY_USE_GITHUB_DATASET_RELEASE else 'local_legacy')
print('[RECOVERY] dataset_release=', DATASET_RELEASE_REPOSITORY, DATASET_RELEASE_TAG)
print('[RECOVERY] evidence_manifest_member=', RECOVERY_EVIDENCE_MANIFEST_RELATIVE_PATH)
print('[RECOVERY] disk_floor_gib=', RECOVERY_MIN_FREE_DISK_GIB, 'reclaim_failed=', RECOVERY_RECLAIM_FAILED_RUN_PAYLOADS)


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb17_cell03_run_recovery.py', globals())
